# Script to query ClinVar's API so we can tell how many dominant pathogenic mutations a gene may contain.

In [1]:
# load libraries
import json
from math import isnan
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import requests
import time
import urllib.request
from urllib.error import HTTPError
import time
from collections import defaultdict
from ast import literal_eval

## Load data

In [ ]:
exon_file='../../data/dnd_ensembl/dnd_ensembl_data.csv'
data=pd.read_csv(exon_file)

In [2]:
data=pd.read_excel('/Users/graceramey/Desktop/UCSF/CapraLab/Conklin_Collabs/Data/GeneSets/dHS/2025-11-13/master_dataframe/dnd_593_2025_12_23.xlsx')
data.columns

Index(['hgnc_symbol', 'GENE ID (HGNC)', 'Approved name', 'DISEASE LABEL',
       'DISEASE ID (MONDO)', 'MOI', 'CLASSIFICATION', 'HI Score', '%HI', 'pLI',
       'LOEUF', 'total_plp', 'missense_plp', 'nonsense_plp', 'ensg', 'chrom',
       's_het', 's_het_lower_CI', 's_het_upper_CI',
       'indel_targetable_pre_NMD_assessment',
       'num_indel_vars_pre_NMD_assessment',
       'num_indel_amenable_vars_inducing_NMD',
       'num_indel_amenable_vars_escaping_NMD',
       'indel_targetable_post_NMD_assessment', 'crisproff_targetable',
       'num_crisproff_vars', 'base_editable', 'num_base_editable_vars',
       'excision_targetable', 'num_excision_vars', 'num_excision_pairs',
       'HPO_term_list', 'indel_pam_targetable', 'crisproff_pam_targetable',
       'base_edit_pam_targetable', 'excision_pam_targetable',
       'hets_across_strats', 'prop_hets_across_strats', 'num_indel_hets',
       'indel_hets_prop', 'num_crisproff_hets', 'crisproff_hets_prop',
       'num_base_edit_hets', 'bas

In [4]:
gene_df=data[['hgnc_symbol']].drop_duplicates() # isolate just the gene names
print(len(gene_df.hgnc_symbol.unique()))

In [5]:
# sort gene names
gene_df.sort_values(by='hgnc_symbol', inplace=True, ignore_index=True)
gene_df

,hgnc_symbol
0,AARS1
1,ABCB6
2,ABCC6
3,ABCC8
4,ABCC9
...,...
588,ZIC1
589,ZMIZ1
590,ZMYND8
591,ZNF292


## Set up results directory

In [ ]:
results_dir = "/wynton/home/capra/gramey02/dnd_project/results"
run_name="RUN_MULTIALLELIC"

## Loop to get number of mutations per gene.
### Note that you'll need to fill in your own requestd API keys.

In [ ]:
# =========================
# ClinVar × OMIM — dominant P/LP variant counts by molecular consequence
# =========================
import os
import re
import json
import time
import requests
import pandas as pd
from collections import Counter, defaultdict, deque

# ---------- endpoints / auth ----------
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
BASE_ESEARCH  = f"{BASE}/esearch.fcgi"
BASE_EPOST    = f"{BASE}/epost.fcgi"
BASE_ESUMMARY = f"{BASE}/esummary.fcgi"
OMIM_ENTRY    = "https://api.omim.org/api/entry"

# Set both of these in your shell profile (~/.zshrc or ~/.bash_profile), e.g.:
#   export NCBI_API_KEY="..."
#   export OMIM_API_KEY="..."
# then restart the kernel so os.environ picks them up. Do not hardcode them
# here — this file gets shared and the keys travel with it.
os.environ["NCBI_API_KEY"]="" ## FILL WITH YOUR NCBI API KEY
NCBI_API_KEY = os.environ.get("NCBI_API_KEY")
os.environ["OMIM_API_KEY"] = "" ## FILL WITH YOUR OMIM API KEY
OMIM_API_KEY = os.environ.get("OMIM_API_KEY")      # <- required
TOOL_NAME  = "capra_clinvar_counts"
TOOL_EMAIL = "" # CAN FILL WITH YOUR EMAIL

if not OMIM_API_KEY:
    raise RuntimeError("Set OMIM_API_KEY in your environment before running.")
if not NCBI_API_KEY:
    print("WARNING: no NCBI_API_KEY — throttled to ~3 req/s. Expect 429s and "
          "empty esummary pages on large genes.", flush=True)

NCBI_MIN_INTERVAL = 0.11 if NCBI_API_KEY else 0.40
OMIM_MIN_INTERVAL = 0.30
ESUMMARY_PAGE_SIZE = 100
EPOST_BATCH = 5000
OMIM_BATCH = 20

# ---------- dominance policy ----------
# UNKNOWN_POLICY: what a variant whose inheritance can't be resolved counts as.
#   "separate"  -> own bucket, excluded from the dominant total  (default)
#   "dominant"  -> folded into the dominant total
#   "not"       -> folded into not-dominant
#   "drop"      -> excluded from total_plp entirely
UNKNOWN_POLICY = "separate"

# MULTI_DISEASE_RULE: variant linked to several diseases with mixed inheritance.
#   "any"  -> dominant if at least one linked disease is dominant  (default)
#   "most" -> dominant if > half of resolved diseases are dominant
#   "all"  -> dominant only if every resolved disease is dominant
MULTI_DISEASE_RULE = "any"

# Union OMIM + OMIMPS rather than preferring one; use MONDO as supplement.
USE_MONDO_FALLBACK = True

# ---------- paths ----------
CHECKPOINT_DIR = (os.path.join(results_dir, run_name, "summary_files/clinvar_muts/checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
PER_GENE_CSV     = os.path.join(CHECKPOINT_DIR, "per_gene_dominance.csv")
PER_VAR_CSV      = os.path.join(CHECKPOINT_DIR, "per_variant_dominance.csv")
ROLLING_CSV      = os.path.join(CHECKPOINT_DIR, "rolling_dominance.csv")
OMIM_CACHE       = os.path.join(CHECKPOINT_DIR, "omim_inheritance_cache.json")
FAILED_GENES_CSV = os.path.join(CHECKPOINT_DIR, "failed_genes.csv")
CHECKPOINT_EVERY_N_GENES = 10

# ---------- small helpers ----------
def _fmt_hms(seconds):
    seconds = int(max(seconds, 0))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:d}:{m:02d}:{s:02d}"

def batched(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

# ---------- P/LP classification filter ----------
LENIENT_PLP = True
STRICT_PLP = {"pathogenic", "likely pathogenic", "pathogenic/likely pathogenic"}

def is_plp(desc: str) -> bool:
    d = (desc or "").strip().lower()
    if not d:
        return False
    if d in STRICT_PLP:
        return True
    if not LENIENT_PLP:
        return False
    lead = d.split(";")[0].split(",")[0].strip()
    return lead in STRICT_PLP

# ---------- consequence categories ----------
CATEGORY_ORDER = [
    "nonsense", "frameshift", "splice_site", "start_lost", "stop_lost",
    "inframe_indel", "missense", "splice_region", "synonymous",
    "utr_5", "utr_3", "non_coding", "intron", "near_gene",
    "no_sequence_alteration", "other", "no_consequence",
]

CONSEQUENCE_PATTERNS = [
    ("splice donor 5th base", "splice_region"),
    ("splice donor",          "splice_site"),
    ("splice acceptor",       "splice_site"),
    ("splice region",         "splice_region"),
    ("splice",                "splice_site"),
    ("stop gained",           "nonsense"),
    ("nonsense",              "nonsense"),
    ("stop lost",             "stop_lost"),
    ("terminator codon",      "stop_lost"),
    ("frameshift",            "frameshift"),
    ("initiator codon",       "start_lost"),
    ("start lost",            "start_lost"),
    ("start retained",        "synonymous"),
    ("inframe",               "inframe_indel"),
    ("in frame",              "inframe_indel"),
    ("missense",              "missense"),
    ("synonymous",            "synonymous"),
    ("stop retained",         "synonymous"),
    ("5 prime utr",           "utr_5"),
    ("3 prime utr",           "utr_3"),
    ("intron",                "intron"),
    ("non-coding transcript", "non_coding"),
    ("non coding transcript", "non_coding"),
    ("nc transcript",         "non_coding"),
    ("upstream",              "near_gene"),
    ("downstream",            "near_gene"),
    ("intergenic",            "near_gene"),
    ("no sequence alteration", "no_sequence_alteration"),
]

UNMAPPED_TERMS = Counter()

def normalize_consequence(raw):
    if isinstance(raw, dict):
        raw = raw.get("type") or raw.get("consequence") or ""
    return str(raw).strip().lower().replace("_", " ")

def categorize(term: str) -> str:
    for pattern, category in CONSEQUENCE_PATTERNS:
        if pattern in term:
            return category
    UNMAPPED_TERMS[term] += 1
    return "other"

def record_consequences(rec):
    raw = list(rec.get("molecular_consequence_list") or [])
    if not raw:
        for vs in rec.get("variation_set") or []:
            if isinstance(vs, dict):
                raw.extend(vs.get("molecular_consequence_list") or [])
    terms = [normalize_consequence(x) for x in raw]
    cats = {categorize(t) for t in terms if t}
    return cats or {"no_consequence"}

def primary_category(cats):
    for c in CATEGORY_ORDER:
        if c in cats:
            return c
    return "other"

# ---------- HTTP ----------
session = requests.Session()
_last_ts = {"ncbi": 0.0, "omim": 0.0}

def _throttle(host):
    interval = NCBI_MIN_INTERVAL if host == "ncbi" else OMIM_MIN_INTERVAL
    elapsed = time.monotonic() - _last_ts[host]
    if elapsed < interval:
        time.sleep(interval - elapsed)
    _last_ts[host] = time.monotonic()

def http_request(method, url, *, host, params=None, data=None,
                 timeout=60, max_tries=8):
    """Throttled request with exponential backoff on 429/5xx/network errors."""
    query, payload = dict(params or {}), dict(data or {})
    if host == "ncbi":
        target = payload if method == "POST" else query
        target["tool"], target["email"] = TOOL_NAME, TOOL_EMAIL
        if NCBI_API_KEY:
            target["api_key"] = NCBI_API_KEY

    delay, last_err = 1.0, None
    for attempt in range(max_tries):
        _throttle(host)
        try:
            r = session.request(method, url, params=query or None,
                                data=payload or None, timeout=timeout)
        except requests.RequestException as e:
            last_err = e
        else:
            if r.status_code < 400:
                return r
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()
            last_err = requests.HTTPError(f"HTTP {r.status_code}")
        print(f"  retry {attempt+1}/{max_tries} on {host} ({last_err}); "
              f"sleeping {delay:.1f}s", flush=True)
        time.sleep(delay)
        delay = min(delay * 2, 30.0)
    raise RuntimeError(f"{url} failed after {max_tries} tries: {last_err}")

# ---------- ClinVar ----------
def esearch_ids(term):
    r = http_request("GET", BASE_ESEARCH, host="ncbi", timeout=30, params={
        "db": "clinvar", "term": term, "retmode": "json", "retmax": 10000})
    data_json = r.json()
    res = data_json.get("esearchresult")
    if res is None:
        raise RuntimeError(f"Malformed esearch response: {data_json}")
    if "ERROR" in res:
        raise RuntimeError(f"esearch error: {res['ERROR']}")
    idlist = res.get("idlist", []) or []
    count = int(res.get("count", len(idlist)))
    if count > len(idlist):
        raise RuntimeError(
            f"esearch truncated: {count} hits reported, {len(idlist)} returned. "
            f"Page this query with retstart or usehistory.")
    return idlist, count

def epost_ids(ids):
    r = http_request("POST", BASE_EPOST, host="ncbi",
                     data={"db": "clinvar", "id": ",".join(ids)})
    txt = r.text
    return (txt.split("<WebEnv>")[1].split("</WebEnv>")[0],
            txt.split("<QueryKey>")[1].split("</QueryKey>")[0])

def iter_summaries(uids, page_size=None, min_page=10):
    """Fetch ClinVar summaries, shrinking retmax when a page comes back empty.

    An empty 200 response means the payload exceeded NCBI's response ceiling
    (large CNV records with long trait lists blow past it) — quarter the page
    size and retry the same offset rather than aborting.
    """
    page_size = page_size or ESUMMARY_PAGE_SIZE
    for post_batch in batched(uids, EPOST_BATCH):
        webenv, query_key = epost_ids(post_batch)
        retstart = 0
        cur = page_size
        while retstart < len(post_batch):
            r = http_request("GET", BASE_ESUMMARY, host="ncbi", params={
                "db": "clinvar", "query_key": query_key, "WebEnv": webenv,
                "retmode": "json", "retstart": retstart, "retmax": cur})
            payload = r.json()
            if "error" in payload:
                raise RuntimeError(f"esummary error: {payload['error']}")
            result = payload.get("result", {})
            page_uids = result.get("uids", []) or []

            if not page_uids:
                if cur > min_page:
                    cur = max(min_page, cur // 4)
                    print(f"  empty page at retstart={retstart}; "
                          f"reducing retmax to {cur}", flush=True)
                    continue
                raise RuntimeError(
                    f"Empty esummary page at retstart={retstart} of "
                    f"{len(post_batch)} UIDs even at retmax={min_page} — "
                    f"aborting rather than silently dropping records.")

            for uid in page_uids:
                rec = result.get(uid, {})
                if isinstance(rec, dict) and "error" not in rec:
                    yield uid, rec
            retstart += len(page_uids)
            # Deliberately not resetting `cur`: if this batch has fat records,
            # the next page probably does too. Avoids repeated failed requests.

def germline_description(rec):
    d = rec.get("germline_classification") or {}
    desc = (d.get("description") or "").strip()
    if not desc:
        d = rec.get("clinical_significance") or {}
        desc = (d.get("description") or "").strip()
    return desc

# ---------- trait -> MIM ids ----------
MIM_RE = re.compile(r"^(PS)?(\d{6})$")

def normalize_mim(raw):
    """Return (mim_number_str, was_series) or None if unparseable."""
    s = str(raw).strip().upper()
    s = s.split(":")[-1]                       # "OMIM:143890" -> "143890"
    m = MIM_RE.match(s)
    if not m:
        return None
    return m.group(2), bool(m.group(1))

def mondo_xrefs(mondo_id):
    """Requires the `mondo` ontology object to be loaded in the session."""
    try:
        term = mondo.get(mondo_id)
    except Exception:
        return set()
    if term is None:
        return set()
    return {getattr(x, "id", "") for x in term.xrefs}

def variant_mim_ids(rec):
    """All MIM numbers linked to a variant's traits. Union of OMIM + OMIMPS,
    supplemented by MONDO->OMIM. Returns (set_of_mims, set_of_series_mims)."""
    trait_set = (rec.get("germline_classification") or {}).get("trait_set") or []
    source_to_ids = defaultdict(list)
    for entry in trait_set:
        for xref in entry.get("trait_xrefs", []) or []:
            db = (xref.get("db_source") or "").strip().upper()
            did = xref.get("db_id")
            if db and did:
                source_to_ids[db].append(did)

    mims, series = set(), set()
    for db in ("OMIM", "OMIMPS"):
        for raw in source_to_ids.get(db, []):
            parsed = normalize_mim(raw)
            if parsed:
                mims.add(parsed[0])
                if parsed[1] or db == "OMIMPS":
                    series.add(parsed[0])

    if USE_MONDO_FALLBACK:
        for mondo_id in source_to_ids.get("MONDO", []):
            for x in mondo_xrefs(mondo_id):
                if str(x).upper().startswith("OMIM"):
                    parsed = normalize_mim(x)
                    if parsed:
                        mims.add(parsed[0])
                        if parsed[1]:
                            series.add(parsed[0])
    return mims, series

# ---------- OMIM inheritance, cached ----------
# cache: mim -> True (dominant) | False (resolved, not dominant) | None (unknown)
if os.path.exists(OMIM_CACHE):
    with open(OMIM_CACHE) as fh:
        omim_cache = json.load(fh)
    print(f"OMIM cache loaded: {len(omim_cache)} MIM numbers", flush=True)
else:
    omim_cache = {}
    print("No OMIM cache found — starting fresh", flush=True)

def save_omim_cache():
    tmp = OMIM_CACHE + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(omim_cache, fh)
    os.replace(tmp, OMIM_CACHE)

def _inheritance_strings(entry):
    """Every inheritance string on an OMIM entry, from all known locations."""
    out = []
    cs = entry.get("clinicalSynopsis") or {}
    if isinstance(cs, dict):
        inh = cs.get("inheritance")
        if inh:
            out.append(str(inh))
        old = cs.get("oldFormat")
        if isinstance(old, dict) and old.get("Inheritance"):
            out.append(str(old["Inheritance"]))

    # phenotypeMapList sits at the top level on phenotype entries and under
    # geneMap on gene entries — check both.
    pm_lists = []
    if entry.get("phenotypeMapList"):
        pm_lists.append(entry["phenotypeMapList"])
    gm = entry.get("geneMap") or {}
    if isinstance(gm, dict) and gm.get("phenotypeMapList"):
        pm_lists.append(gm["phenotypeMapList"])
    for pm_list in pm_lists:
        for item in pm_list or []:
            pi = (item.get("phenotypeMap") or {}).get("phenotypeInheritance")
            if pi:
                out.append(str(pi))
    return out

def resolve_mims(mims):
    """Populate omim_cache for any MIM not already present."""
    missing = sorted(m for m in mims if m not in omim_cache)
    if not missing:
        return
    for batch in batched(missing, OMIM_BATCH):
        r = http_request("GET", OMIM_ENTRY, host="omim", timeout=30, params={
            "mimNumber": ",".join(batch),
            "include": ["clinicalSynopsis", "geneMap"],
            "format": "json",
            "apiKey": OMIM_API_KEY,
        })
        entry_list = (r.json().get("omim") or {}).get("entryList") or []
        for wrapper in entry_list:
            entry = wrapper.get("entry") or {}
            mim = str(entry.get("mimNumber", "")).strip()
            if not mim:
                continue
            strings = _inheritance_strings(entry)
            if not strings:
                omim_cache[mim] = None          # entry exists, no inheritance
            else:
                omim_cache[mim] = any(
                    "autosomal dominant" in s.lower() for s in strings)
        # requested but not returned -> withdrawn/invalid; cache as unknown
        for mim in batch:
            omim_cache.setdefault(mim, None)
    save_omim_cache()

def dominance_from_mims(mims):
    """-> 'dominant' | 'not_dominant' | 'unknown'"""
    if not mims:
        return "unknown"
    vals = [omim_cache.get(m) for m in mims]
    known = [v for v in vals if v is not None]
    if not known:
        return "unknown"
    if MULTI_DISEASE_RULE == "any":
        return "dominant" if any(known) else "not_dominant"
    if MULTI_DISEASE_RULE == "all":
        return "dominant" if all(known) else "not_dominant"
    if MULTI_DISEASE_RULE == "most":
        return "dominant" if sum(known) * 2 > len(known) else "not_dominant"
    raise ValueError(f"Unknown MULTI_DISEASE_RULE: {MULTI_DISEASE_RULE}")

def counts_as_dominant(status):
    if status in ("dominant_clinvar", "dominant_omim"):
        return True
    if status == "unknown":
        return UNKNOWN_POLICY == "dominant"
    return False

def counts_in_total(status):
    return not (status == "unknown" and UNKNOWN_POLICY == "drop")

# ---------- output schema ----------
DOM_STATES = ["dominant_clinvar", "dominant_omim", "not_dominant", "unknown"]
GENE_COLUMNS = (
    ["counter", "gene", "esearch_hits_broad", "esearch_hits_moi_dom",
     "total_plp", "total_dominant"]
    + [f"n_{s}" for s in DOM_STATES]
    + [f"any_{c}" for c in CATEGORY_ORDER]
    + [f"primary_{c}" for c in CATEGORY_ORDER]
)
VAR_COLUMNS = ["gene", "uid", "germline_classification", "dominance_status",
               "is_dominant", "primary_consequence", "all_consequences",
               "mim_ids", "series_mim_ids"]
FAILED_COLUMNS = ["gene", "counter", "esearch_hits_broad",
                  "n_staged_before_failure", "error"]

def append_rows(path, rows, columns):
    if not rows:
        return
    pd.DataFrame(rows, columns=columns).to_csv(
        path, mode="a", header=not os.path.exists(path), index=False)

def load_gene_checkpoint():
    if not os.path.exists(PER_GENE_CSV):
        return [], set()
    df = pd.read_csv(PER_GENE_CSV)
    if df.empty:
        return [], set()
    df = df.drop_duplicates(subset=["gene"], keep="last").sort_values("counter")
    return df.to_dict("records"), set(df["gene"].astype(str))

# ---------- gene list ----------
gene_symbols = pd.Series(data.hgnc_symbol.unique()).dropna().astype(str).str.strip()
gene_symbols = gene_symbols[gene_symbols != ""]
gene_symbols = gene_symbols.drop_duplicates().tolist()

# Stable positional ID, independent of what's left to do this run.
gene_index = {g: i for i, g in enumerate(gene_symbols)}

# ---------- resume ----------
gene_rows, done_genes = load_gene_checkpoint()

todo = [g for g in gene_symbols if g not in done_genes]
n_todo = len(todo)
print(f"{len(gene_symbols)} genes total | {len(gene_symbols) - n_todo} already "
      f"done | {n_todo} to process", flush=True)

run_start = time.monotonic()
recent = deque(maxlen=25)   # rolling window for ETA
failed_genes = []

# ---------- main loop ----------
for i, gene in enumerate(todo, start=1):
    counter = gene_index[gene]
    gene_start = time.monotonic()

    if recent:
        eta = (sum(recent) / len(recent)) * (n_todo - i + 1)
        eta_str = _fmt_hms(eta)
    else:
        eta_str = "--:--:--"

    print(f"[{i}/{n_todo} {100.0*(i-1)/n_todo:5.1f}%] {gene:<12} "
          f"elapsed {_fmt_hms(time.monotonic() - run_start)}  ETA {eta_str}",
          flush=True)

    clinsig = ('(pathogenic[clinsig] OR "likely pathogenic"[clinsig] OR '
               '"pathogenic/likely pathogenic"[clinsig])')

    # narrow: ClinVar-annotated autosomal dominant. Strict subset of broad.
    moi_ids, moi_count = esearch_ids(
        f'{gene}[gene] AND "moi autosomal dominant"[prop] AND {clinsig}')
    moi_dom_uids = set(moi_ids)

    # broad: every P/LP variant for the gene. One esummary pass over this.
    all_ids, broad_count = esearch_ids(f'{gene}[gene] AND {clinsig}')

    # --- pass 1: pull records, collect MIMs needing resolution ---
    staged = []
    mims_to_resolve = set()
    try:
        for uid, rec in iter_summaries(all_ids):
            desc = germline_description(rec)
            if not is_plp(desc):
                continue
            cats = record_consequences(rec)
            if uid in moi_dom_uids:
                staged.append((uid, desc, cats, set(), set(), True))
            else:
                mims, series = variant_mim_ids(rec)
                mims_to_resolve |= mims
                staged.append((uid, desc, cats, mims, series, False))
    except RuntimeError as e:
        # Partial data only — discard it and do NOT checkpoint this gene,
        # otherwise the resume logic treats an undercount as complete.
        print(f"      SKIPPING {gene}: {e}", flush=True)
        failed_genes.append({"gene": gene, "counter": counter,
                             "esearch_hits_broad": broad_count,
                             "n_staged_before_failure": len(staged),
                             "error": str(e)})
        append_rows(FAILED_GENES_CSV, [failed_genes[-1]], FAILED_COLUMNS)
        continue

    # --- one batched OMIM round-trip per gene, cache-backed ---
    resolve_mims(mims_to_resolve)

    # --- pass 2: assign dominance, count ---
    any_counts, primary_counts, state_counts = Counter(), Counter(), Counter()
    total_plp = total_dominant = 0
    var_rows = []

    for uid, desc, cats, mims, series, by_clinvar in staged:
        if by_clinvar:
            status = "dominant_clinvar"
        else:
            resolved = dominance_from_mims(mims)
            status = {"dominant": "dominant_omim",
                      "not_dominant": "not_dominant",
                      "unknown": "unknown"}[resolved]

        state_counts[status] += 1
        if not counts_in_total(status):
            continue
        total_plp += 1

        is_dom = counts_as_dominant(status)
        if is_dom:
            total_dominant += 1
            for c in cats:
                any_counts[c] += 1
            primary_counts[primary_category(cats)] += 1

        var_rows.append({
            "gene": gene,
            "uid": uid,
            "germline_classification": desc,
            "dominance_status": status,
            "is_dominant": is_dom,
            "primary_consequence": primary_category(cats),
            "all_consequences": ";".join(sorted(cats)),
            "mim_ids": ";".join(sorted(mims)),
            "series_mim_ids": ";".join(sorted(series)),
        })

    row = {
        "counter": counter, "gene": gene,
        "esearch_hits_broad": broad_count,
        "esearch_hits_moi_dom": moi_count,
        "total_plp": total_plp,
        "total_dominant": total_dominant,
    }
    for s in DOM_STATES:
        row[f"n_{s}"] = state_counts.get(s, 0)
    for c in CATEGORY_ORDER:
        row[f"any_{c}"] = any_counts.get(c, 0)
        row[f"primary_{c}"] = primary_counts.get(c, 0)

    append_rows(PER_VAR_CSV, var_rows, VAR_COLUMNS)
    append_rows(PER_GENE_CSV, [row], GENE_COLUMNS)
    gene_rows.append(row)
    done_genes.add(gene)

    took = time.monotonic() - gene_start
    recent.append(took)
    print(f"      {total_plp} P/LP | {total_dominant} dom "
          f"({state_counts['dominant_clinvar']} ClinVar / "
          f"{state_counts['dominant_omim']} OMIM) | "
          f"{state_counts['not_dominant']} not-dom | "
          f"{state_counts['unknown']} unk | "
          f"{len(omim_cache)} MIMs cached | {took:.1f}s", flush=True)

    if len(gene_rows) % CHECKPOINT_EVERY_N_GENES == 0:
        pd.DataFrame(gene_rows, columns=GENE_COLUMNS).to_csv(ROLLING_CSV, index=False)

pd.DataFrame(gene_rows, columns=GENE_COLUMNS).to_csv(ROLLING_CSV, index=False)
save_omim_cache()

print(f"\nDone in {_fmt_hms(time.monotonic() - run_start)}.")
print(f"  Per-gene:    {PER_GENE_CSV}")
print(f"  Per-variant: {PER_VAR_CSV}")
print(f"  OMIM cache:  {OMIM_CACHE}  ({len(omim_cache)} MIMs)")

if failed_genes:
    print(f"\n{len(failed_genes)} genes skipped — see {FAILED_GENES_CSV}")
    for f in failed_genes:
        print(f"  {f['gene']} ({f['esearch_hits_broad']} hits)")
    print("  These were not checkpointed and will be retried on the next run.")

if UNMAPPED_TERMS:
    print("\nUnmapped consequence terms (counted as 'other'):")
    for t, n in UNMAPPED_TERMS.most_common():
        print(f"  {n:6d}  {t}")